# Clustering

Use the **bao** kernel in your allocated HPC remote desktop. Edit these cells and the reusable functions in `pzbao/diagnostics/clustering.py`.

These small synthetic examples illustrate the interface; they are not BAO results. Scientific interpretation and acceptance remain yours.


In [ ]:
from pathlib import Path
import numpy as np
from pzbao.diagnostics import clustering as diagnostics
from pzbao.config import load_analysis_config
from pzbao.config.stages import stage_document
from pzbao.run import load_run, stage_units

RUN_ROOT = None  # Set to the named run directory you want to inspect.
run = load_run(RUN_ROOT) if RUN_ROOT is not None else None
config = run["config"] if run is not None else load_analysis_config()


## Settings and saved products

This is the section used by `bao cluster`. Edit a complete stage YAML or a combined experiment YAML. CLI runs freeze scientific settings; use a new named run for a changed experiment.


In [ ]:
settings = stage_document(config, "cluster")
settings


In [ ]:
if run is not None:
    units = stage_units("cluster", config, run["metadata"]["workload"])
    products = list((run["root"] / "cluster" / "units").glob("*/result.yaml"))
    print(f"{len(products)} saved units; {len(units)} configured units")


## Explore and plot

Replace the illustrative arrays with saved arrays from your chosen unit. Plotting functions accept an existing axis and return their data for inspection.


In [ ]:
s = np.linspace(50.0, 150.0, 20)
multipoles = np.stack([np.sin(s / 10.0) / s, np.cos(s / 10.0) / s])
ax, values = diagnostics.plot_multipoles(s, multipoles, [0, 2])
values["values"].shape


## Load one saved result

Set `LOAD_SAVED = True` after selecting your run and unit. These cells only read saved products; missing prerequisites remain visible as errors. The clustering example reads xi; P(k)-only runs can be inspected through the unit's `measurements.json` and named power NPZ files.


In [ ]:
LOAD_SAVED = False
SAVED_UNIT_INDEX = 0  # Choose the configured unit/route to inspect.
if LOAD_SAVED:
    if run is None:
        raise ValueError("Set RUN_ROOT before loading saved products")
    from pzbao.clustering import read_measurements
    from pzbao.run import routes_for

    route = routes_for(config, run["metadata"]["workload"])[SAVED_UNIT_INDEX]
    records = read_measurements(run.get("source_run", run), route)
    record = next(row for row in records if row["identity"]["cap"] == "NGC+SGC")
    MULTIPOLES = config["measurement"]["xi"]["multipoles"]
    coordinate, multipoles = record["estimator"](ells=MULTIPOLES, return_sep=True)
    saved_ax, saved_values = diagnostics.plot_multipoles(coordinate, multipoles, MULTIPOLES)


## Interpretation

Record the chosen unit, comparisons, missing or failed products, relevant flags, and your scientific question here. Numerical stopping and report publication do not establish a validation verdict.
